# Estimation and visualization of Kernel Treatment Effect (KTE)

## Load data and settings

In [1]:
%cd ../src

c:\Users\Caleb\HeartstepsDNN\Heartsteps-DNN\src


c:\Users\Caleb\HeartstepsDNN\Heartsteps-DNN\dnn_env\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
%load_ext autoreload
%autoreload 2

In [62]:
# local modules
import dnn
import cross_validate_dnn as cv_dnn
import scaler_methods

# packages
import numpy as np
import pandas as pd
from itertools import chain
import itertools
from matplotlib.pyplot import matshow
import matplotlib.pyplot as plt
import matplotlib as mpl
import math
import random

import scipy
from scipy.stats import kstest
mpl.rcParams.update(mpl.rcParamsDefault)

In [63]:
plt.rc('font', family='serif')
plt.rc('text', usetex=False)
plt.rc('axes', axisbelow=True)
label_size = 15
mpl.rcParams['xtick.labelsize'] = label_size 
mpl.rcParams['ytick.labelsize'] = label_size 
mpl.rcParams['axes.labelsize'] = label_size
mpl.rcParams['axes.titlesize'] = label_size
mpl.rcParams['figure.titlesize'] = label_size
mpl.rcParams['lines.markersize'] = label_size
mpl.rcParams['grid.linewidth'] = 2.5
mpl.rcParams['legend.fontsize'] = label_size


lss = ['--',  ':', '-.', '-', '--', '-.', ':', '-', '--', '-.', ':', '-']
mss = ['x', 'o', '>', 's', 'D', '>', 'D', '>', 's', 'o', 'D']

In [12]:
# set up
A_hs = np.load("../data_checkpoints/heartsteps_mask.npy")
M_hs = np.load("../data_checkpoints/heartsteps_data.npy")

# required for Wasserstein
M_hs = np.sort(M_hs, axis = -1)

# dimension of the data is 1
M_hs = M_hs[:, :, :, None]

# truncate the data as most users have had 40 days
M_hs = M_hs[:, 0:200]
A_hs = A_hs[:, 0:200]

N, T, n, d = M_hs.shape

# LOG TRANSFORMED HERE!!!!!!!
M_hs = np.log(M_hs + 1)



In [17]:
diffs = dnn.get_col_differences(M_hs, A_zero, metric = "wasserstein2", kernel = "exponential")
np.nanmax(diffs[~np.isinf(diffs)])

c:\Users\Caleb\HeartstepsDNN\Heartsteps-DNN\src\dnn.py:256: RuntimeWarning: Mean of empty slice
  diffs = np.nanmean((Z_cp.swapaxes(0, 1)[:, None, :,  :] - Z_cp.swapaxes(0, 1))**2, axis = -1)
c:\Users\Caleb\HeartstepsDNN\Heartsteps-DNN\src\dnn.py:259: RuntimeWarning: Mean of empty slice
  diffs = np.nanmean(diffs, axis = -1)


np.float64(18.161385634930195)

In [23]:
len(A_zero[A_zero == 1])

4511

## Estimation

We wish to compute the counterfactual estimates for every entry in the matrix for both with the treatment (A_hs = 1) and without the treatment (A_hs = 0). We ignore entries with A_hs = 2. 

In [27]:
A_zero = A_hs.copy()
A_zero[A_hs == 1] = 0
A_zero[A_hs == 0] = 1

split_A = np.split(A_hs, 2, axis = 1)
split_M = np.split(M_hs, 2, axis = 1)

split_A_zero = np.split(A_zero, 2, axis = 1)


width = len(A_hs[0])

# A1 represents the estimate of the entry if the user had received treatment
# A0 represents the estimate of the entry if the user had not received treatment
# We compute these estimates for every entry regardless of its mask value in the
# actual data set.

ws2_estimates_A1 = np.full([N, T, n], np.nan)
mmd_estimates_exp_A1 = [None] * N
mmd_estimates_sq_A1 = [None] * N

ws2_estimates_A0 = np.full([N, T, n], np.nan)
mmd_estimates_exp_A0 = [None] * N
mmd_estimates_sq_A0 = [None] * N

nn_ws2_A1 = np.full([N, T], np.nan)
nn_mmd_exp_A1 = np.full([N, T], np.nan)
nn_mmd_sq_A1 = np.full([N, T], np.nan)

nn_ws2_A0 = np.full([N, T], np.nan)
nn_mmd_exp_A0 = np.full([N, T], np.nan)
nn_mmd_sq_A0 = np.full([N, T], np.nan)

eta_stars_A1 = np.full([37, 6], np.nan)
eta_stars_A0 = np.full([37, 6], np.nan)

neighbors_ws2_A1 = [None] * N
neighbors_ws2_A0 = [None] * N

neighbors_mmd_exp_A1 = [None] * N
neighbors_mmd_exp_A0 = [None] * N

neighbors_mmd_sq_A1 = [None] * N
neighbors_mmd_sq_A0 = [None] * N

for i in range(len(A_hs)):
  print("User " + str(i))

  # sample split for estimating A1
  dist_A = A_hs.copy()
  dist_A[i] = 0

  wasserstein_diffs_A1 = dnn.get_col_differences(M_hs, dist_A, metric = "wasserstein2", kernel = None)
  mmd_exp_diffs_A1 = dnn.get_col_differences(M_hs, dist_A, metric = "mmd", kernel = "exponential")
  mmd_sq_diffs_A1 = dnn.get_col_differences(M_hs, dist_A, metric = "mmd", kernel = "square")

  # partially invert the mask
  A_zero_dist = A_zero.copy() 
  A_zero_dist[i] = 0

  wasserstein_diffs_A0 = dnn.get_col_differences(M_hs, A_zero_dist, metric = "wasserstein2", kernel = None)
  mmd_exp_diffs_A0 = dnn.get_col_differences(M_hs, A_zero_dist, metric = "mmd", kernel = "exponential")
  mmd_sq_diffs_A0 = dnn.get_col_differences(M_hs, A_zero_dist, metric = "mmd", kernel = "square")

  s_ws2_A1 = np.split(wasserstein_diffs_A1, 2, axis = 0)
  diffs_ws2_front_A1 = np.split(s_ws2_A1[0], 2, axis = 1)[0]
  diffs_ws2_back_A1 = np.split(s_ws2_A1[1], 2, axis = 1)[1]

  s_mmd_exp_A1 = np.split(mmd_exp_diffs_A1, 2, axis = 0)
  diffs_mmd_exp_front_A1 = np.split(s_mmd_exp_A1[0], 2, axis = 1)[0]
  diffs_mmd_exp_back_A1 = np.split(s_mmd_exp_A1[1], 2, axis = 1)[1]

  s_mmd_sq_A1 = np.split(mmd_sq_diffs_A1, 2, axis = 0)
  diffs_mmd_sq_front_A1 = np.split(s_mmd_sq_A1[0], 2, axis = 1)[0]
  diffs_mmd_sq_back_A1 = np.split(s_mmd_sq_A1[1], 2, axis = 1)[1]

  s_ws2_A0 = np.split(wasserstein_diffs_A0, 2, axis = 0)
  diffs_ws2_front_A0 = np.split(s_ws2_A0[0], 2, axis = 1)[0]
  diffs_ws2_back_A0= np.split(s_ws2_A0[1], 2, axis = 1)[1]

  s_mmd_exp_A0 = np.split(mmd_exp_diffs_A0, 2, axis = 0)
  diffs_mmd_exp_front_A0 = np.split(s_mmd_exp_A0[0], 2, axis = 1)[0]
  diffs_mmd_exp_back_A0 = np.split(s_mmd_exp_A0[1], 2, axis = 1)[1]

  s_mmd_sq_A0 = np.split(mmd_sq_diffs_A0, 2, axis = 0)
  diffs_mmd_sq_front_A0 = np.split(s_mmd_sq_A0[0], 2, axis = 1)[0]
  diffs_mmd_sq_back_A0 = np.split(s_mmd_sq_A0[1], 2, axis = 1)[1]
  
  # four eta searches per row, two using the front half (A1 and A0)
  # two using the back half (A1 and A0)

  front_A_hs = split_A[0]
  back_A_hs = split_A[1]
  front_A_zero = split_A_zero[0]
  back_A_zero = split_A_zero[1]

  front_M_hs = split_M[0]
  back_M_hs = split_M[1]

  eta_star_ws2_front_A1, _ = cv_dnn.search_eta(front_M_hs, front_A_hs, 5, i, 
                                         diffs_ws2_front_A1, dnn.estimate_row, 
                                         metric="wasserstein2", 
                                         range_min = 0, range_max = 50)
  eta_stars_A1[i, 0] = eta_star_ws2_front_A1
  eta_star_ws2_back_A1, _ = cv_dnn.search_eta(back_M_hs, back_A_hs, 5, i,
                                        diffs_ws2_back_A1, dnn.estimate_row,
                                        metric = "wasserstein2",
                                        range_min = 0, range_max = 50)
  eta_stars_A1[i, 1] = eta_star_ws2_back_A1
  eta_star_mmd_exp_front_A1, _ = cv_dnn.search_eta(front_M_hs, front_A_hs, 5, i, 
                                         diffs_mmd_exp_front_A1, dnn.estimate_row, 
                                         metric="mmd", kernel = "exponential", 
                                         range_min = 0, range_max = 2.0)
  eta_stars_A1[i, 2] = eta_star_mmd_exp_front_A1
  eta_star_mmd_exp_back_A1, _ = cv_dnn.search_eta(back_M_hs, back_A_hs, 5, i,
                                        diffs_mmd_exp_back_A1, dnn.estimate_row,
                                        metric = "mmd", kernel = "exponential",
                                        range_min = 0, range_max = 2.0)
  eta_stars_A1[i, 3] = eta_star_mmd_exp_back_A1
  eta_star_mmd_sq_front_A1, _ = cv_dnn.search_eta(front_M_hs, front_A_hs, 5, i, 
                                         diffs_mmd_sq_front_A1, dnn.estimate_row, 
                                         metric="mmd", kernel = "square", 
                                         range_min = 0, range_max = 2500)
  eta_stars_A1[i, 4] = eta_star_mmd_sq_front_A1
  eta_star_mmd_sq_back_A1,_ = cv_dnn.search_eta(back_M_hs, back_A_hs, 5, i,
                                        diffs_mmd_sq_back_A1, dnn.estimate_row,
                                        metric = "mmd", kernel = "square",
                                        range_min = 0, range_max = 2500)
  eta_stars_A1[i, 5] = eta_star_mmd_sq_back_A1


  
  eta_star_ws2_front_A0, _ = cv_dnn.search_eta(front_M_hs, front_A_zero, 5, i, 
                                         diffs_ws2_front_A0, dnn.estimate_row, 
                                         metric="wasserstein2", 
                                         range_min = 0, range_max = 50)
  eta_stars_A0[i, 0] = eta_star_ws2_front_A0
  eta_star_ws2_back_A0, _ = cv_dnn.search_eta(back_M_hs, back_A_zero, 5, i,
                                        diffs_ws2_back_A0, dnn.estimate_row,
                                        metric = "wasserstein2",
                                        range_min = 0, range_max = 50)
  eta_stars_A0[i, 1] = eta_star_ws2_back_A0
  eta_star_mmd_exp_front_A0, _ = cv_dnn.search_eta(front_M_hs, front_A_zero, 5, i, 
                                         diffs_mmd_exp_front_A1, dnn.estimate_row, 
                                         metric="mmd", kernel = "exponential", 
                                         range_min = 0, range_max = 2.0)
  eta_stars_A0[i, 2] = eta_star_mmd_exp_front_A0
  eta_star_mmd_exp_back_A0, _ = cv_dnn.search_eta(back_M_hs, back_A_zero, 5, i,
                                        diffs_mmd_exp_back_A0, dnn.estimate_row,
                                        metric = "mmd", kernel = "exponential",
                                        range_min = 0, range_max = 2.0)
  eta_stars_A0[i, 3] = eta_star_mmd_exp_back_A0
  eta_star_mmd_sq_front_A0, _ = cv_dnn.search_eta(front_M_hs, front_A_zero, 5, i, 
                                         diffs_mmd_sq_front_A0, dnn.estimate_row, 
                                         metric="mmd", kernel = "square", 
                                         range_min = 0, range_max = 2500)
  eta_stars_A0[i, 4] = eta_star_mmd_sq_front_A0
  eta_star_mmd_sq_back_A0,_ = cv_dnn.search_eta(back_M_hs, back_A_zero, 5, i,
                                        diffs_mmd_sq_back_A0, dnn.estimate_row,
                                        metric = "mmd", kernel = "square",
                                        range_min = 0, range_max = 2500)
  eta_stars_A0[i, 5] = eta_star_mmd_sq_back_A0


  # estimate first half using eta from second
  # test_inds_A1 = np.nonzero(A_hs[i] == 1)[0]
  # front_test_inds_A1 = test_inds_A1[test_inds_A1 < (width / 2)]
  # back_test_inds = test_inds_A1[test_inds_A1 >= (width / 2)]

  # test_inds_A1 = np.nonzero(A_hs[i] == 1)[0]
  # front_test_inds_A1 = test_inds_A1[test_inds_A1 < (width / 2)]
  # back_test_inds = test_inds_A1[test_inds_A1 >= (width / 2)]

  # if the unit is available, it is a test indice

  test_inds_all = np.nonzero(A_hs[i] != 2)[0]
  front_test_inds_all = test_inds_all[test_inds_all < (width / 2)]
  back_test_inds_all = test_inds_all[test_inds_all >= (width / 2)]
  print("Estimating unit " + str(i))

  estimate_front_ws2_A1, nn_front_ws2_A1, neighbors_front_ws2_A1 = dnn.estimate_row(M_hs, 
                                                                                         A_hs, i, front_test_inds_all, 
                                                                                         wasserstein_diffs_A1, eta_star_ws2_back_A1, 
                                                                                         "wasserstein2", ret_nn = True)
  estimate_front_ws2_A1 = estimate_front_ws2_A1[:T // 2]
  nn_front_ws2_A1 = nn_front_ws2_A1[:T//2]
  neigbors_front_ws2_A1 = neighbors_front_ws2_A1[:T // 2]

  estimate_back_ws2_A1, nn_back_ws2_A1, neighbors_back_ws2_A1 = dnn.estimate_row(M_hs, 
                                                                                 A_hs, i, back_test_inds_all, 
                                                                                 wasserstein_diffs_A1, eta_star_ws2_front_A1, 
                                                                                 "wasserstein2", ret_nn = True)
  estimate_back_ws2_A1 = estimate_back_ws2_A1[T // 2 : ]
  nn_back_ws2_A1 = nn_back_ws2_A1[T//2 : ]
  neighbors_back_ws2_A1 = neighbors_back_ws2_A1[T//2 :]

  estimate_front_mmd_exp_A1, nn_front_mmd_exp_A1, neighbors_front_mmd_exp_A1 = dnn.estimate_row(M_hs, 
                                                                                                A_hs, i, front_test_inds_all, 
                                                                                                mmd_exp_diffs_A1, eta_star_mmd_exp_back_A1,
                                                                                                "mmd", ret_nn = True)
  estimate_front_mmd_exp_A1 = estimate_front_mmd_exp_A1[:T // 2 ]
  nn_front_mmd_exp_A1 = nn_front_mmd_exp_A1[:T // 2 ]
  neighbors_front_mmd_exp_A1 = neighbors_front_mmd_exp_A1[:T//2]

  estimate_back_mmd_exp_A1, nn_back_mmd_exp_A1, neighbors_back_mmd_exp_A1 = dnn.estimate_row(M_hs, 
                                                                                         A_hs, i, back_test_inds_all,
                                                                                         mmd_exp_diffs_A1, eta_star_mmd_exp_front_A1,
                                                                                         "mmd", ret_nn = True)
  estimate_back_mmd_exp_A1 = estimate_back_mmd_exp_A1[T // 2 :]
  nn_back_mmd_exp_A1 = nn_back_mmd_exp_A1[T // 2 :]
  neighbors_back_mmd_exp_A1 = neighbors_back_mmd_exp_A1[T // 2 :]

  estimate_front_mmd_sq_A1, nn_front_mmd_sq_A1, neighbors_front_mmd_sq_A1 = dnn.estimate_row(M_hs, A_hs, i, front_test_inds_all,
                                            mmd_sq_diffs_A1, eta_star_mmd_sq_back_A1,
                                            "mmd", ret_nn = True)
  estimate_front_mmd_sq_A1 = estimate_front_mmd_sq_A1[:T // 2]
  nn_front_mmd_sq_A1 = nn_front_mmd_sq_A1[:T//2]
  neighbors_front_mmd_sq_A1 = neighbors_front_mmd_sq_A1[:T // 2]

  estimate_back_mmd_sq_A1, nn_back_mmd_sq_A1, neighbors_back_mmd_sq_A1 = dnn.estimate_row(M_hs, A_hs, i, back_test_inds_all,
                                            mmd_sq_diffs_A1, eta_star_mmd_sq_front_A1,
                                            "mmd", ret_nn = True)
  estimate_back_mmd_sq_A1 = estimate_back_mmd_sq_A1[T // 2 :]
  nn_back_mmd_sq_A1 = nn_back_mmd_sq_A1[T // 2 :]
  neighbors_front_mmd_sq_A1 = neighbors_back_mmd_sq_A1[T // 2 :]

# ----------------------------------- This is A0 territory

  estimate_front_ws2_A0, nn_front_ws2_A0, neighbors_front_ws2_A0 = dnn.estimate_row(M_hs, 
                                                                                          A_zero, i, front_test_inds_all,
                                                                                          wasserstein_diffs_A0, eta_star_ws2_back_A0, 
                                                                                         "wasserstein2", ret_nn = True)
  estimate_front_ws2_A0 = estimate_front_ws2_A0[:T // 2]
  nn_front_ws2_A0 = nn_front_ws2_A0[:T//2]
  neigbors_front_ws2_A0 = neighbors_front_ws2_A0[:T // 2]

  estimate_back_ws2_A0, nn_back_ws2_A0, neighbors_back_ws2_A0 = dnn.estimate_row(M_hs, 
                                                                                 A_zero, i, back_test_inds_all, 
                                                                                 wasserstein_diffs_A0, eta_star_ws2_front_A0, 
                                                                                 "wasserstein2", ret_nn = True)
  estimate_back_ws2_A0 = estimate_back_ws2_A0[T // 2 : ]
  nn_back_ws2_A0 = nn_back_ws2_A0[T//2 : ]
  neighbors_back_ws2_A0 = neighbors_back_ws2_A0[T//2 :]

  estimate_front_mmd_exp_A0, nn_front_mmd_exp_A0, neighbors_front_mmd_exp_A0 = dnn.estimate_row(M_hs, 
                                                                                                A_zero, i, front_test_inds_all, 
                                                                                                mmd_exp_diffs_A0, eta_star_mmd_exp_back_A0,
                                                                                                "mmd", ret_nn = True)
  estimate_front_mmd_exp_A0 = estimate_front_mmd_exp_A0[:T // 2 ]
  nn_front_mmd_exp_A0 = nn_front_mmd_exp_A0[:T // 2 ]
  neighbors_front_mmd_exp_A0 = neighbors_front_mmd_exp_A0[:T//2]

  estimate_back_mmd_exp_A0, nn_back_mmd_exp_A0, neighbors_back_mmd_exp_A0 = dnn.estimate_row(M_hs, 
                                                                                         A_zero, i, back_test_inds_all,
                                                                                         mmd_exp_diffs_A0, eta_star_mmd_exp_front_A0,
                                                                                         "mmd", ret_nn = True)
  estimate_back_mmd_exp_A0 = estimate_back_mmd_exp_A0[T // 2 :]
  nn_back_mmd_exp_A0 = nn_back_mmd_exp_A0[T // 2 :]
  neighbors_back_mmd_exp_A0 = neighbors_back_mmd_exp_A0[T // 2 :]

  estimate_front_mmd_sq_A0, nn_front_mmd_sq_A0, neighbors_front_mmd_sq_A0 = dnn.estimate_row(M_hs, A_zero, i, front_test_inds_all,
                                            mmd_sq_diffs_A0, eta_star_mmd_sq_back_A0,
                                            "mmd", ret_nn = True)
  estimate_front_mmd_sq_A0 = estimate_front_mmd_sq_A0[:T // 2]
  nn_front_mmd_sq_A0 = nn_front_mmd_sq_A0[:T//2]
  neighbors_front_mmd_sq_A0 = neighbors_front_mmd_sq_A0[:T // 2]

  estimate_back_mmd_sq_A0, nn_back_mmd_sq_A0, neighbors_back_mmd_sq_A0 = dnn.estimate_row(M_hs, A_zero, i, back_test_inds_all,
                                            mmd_sq_diffs_A0, eta_star_mmd_sq_front_A0,
                                            "mmd", ret_nn = True)
  estimate_back_mmd_sq_A0 = estimate_back_mmd_sq_A0[T // 2 :]
  nn_back_mmd_sq_A0 = nn_back_mmd_sq_A0[T // 2 :]
  neighbors_front_mmd_sq_A0 = neighbors_back_mmd_sq_A0[T // 2 :]
  
  # concat estimates together
  est_row_ws2_A1 = np.concatenate((estimate_front_ws2_A1, estimate_back_ws2_A1), axis = 0)
  est_row_mmd_exp_A1 = estimate_front_mmd_exp_A1 + estimate_back_mmd_exp_A1
  est_row_mmd_sq_A1 = estimate_front_mmd_sq_A1 + estimate_back_mmd_sq_A1

  row_nn_ws2_A1 = np.concatenate((nn_front_ws2_A1, nn_back_ws2_A1), axis = 0)
  row_nn_mmd_exp_A1 = np.concatenate((nn_front_mmd_exp_A1, nn_back_mmd_exp_A1), axis = 0)
  row_nn_mmd_sq_A1 = np.concatenate((nn_front_mmd_sq_A1, nn_back_mmd_sq_A1), axis = 0)

  row_neighbors_ws2_A1 = neighbors_front_ws2_A1 + neighbors_back_ws2_A1
  row_neighbors_mmd_exp_A1 = neighbors_front_mmd_exp_A1 + neighbors_back_mmd_exp_A1
  row_neighbors_mmd_sq_A1 = neighbors_front_mmd_sq_A1 + neighbors_back_mmd_sq_A1


  # REMEMBER TO HANDLE RETURNING NEIGHBORS !

  ws2_estimates_A1[i] = est_row_ws2_A1
  mmd_estimates_exp_A1[i] = est_row_mmd_exp_A1
  mmd_estimates_sq_A1[i] = est_row_mmd_sq_A1

  nn_ws2_A1[i] = row_nn_ws2_A1
  nn_mmd_exp_A1[i] = row_nn_mmd_exp_A1
  nn_mmd_sq_A1[i] = row_nn_mmd_sq_A1

  neighbors_ws2_A1[i] = row_neighbors_ws2_A1
  neighbors_mmd_exp_A1[i] = row_neighbors_mmd_exp_A1
  neighbors_mmd_sq_A1[i] = row_neighbors_mmd_sq_A1
  
  # ----- A0 stuff
  est_row_ws2_A0 = np.concatenate((estimate_front_ws2_A0, estimate_back_ws2_A0), axis = 0)
  est_row_mmd_exp_A0 = estimate_front_mmd_exp_A0 + estimate_back_mmd_exp_A0
  est_row_mmd_sq_A0 = estimate_front_mmd_sq_A0 + estimate_back_mmd_sq_A0

  row_nn_ws2_A0 = np.concatenate((nn_front_ws2_A0, nn_back_ws2_A0), axis = 0)
  row_nn_mmd_exp_A0 = np.concatenate((nn_front_mmd_exp_A0, nn_back_mmd_exp_A0), axis = 0)
  row_nn_mmd_sq_A0 = np.concatenate((nn_front_mmd_sq_A0, nn_back_mmd_sq_A0), axis = 0)

  row_neighbors_ws2_A0 = neighbors_front_ws2_A0 + neighbors_back_ws2_A0
  row_neighbors_mmd_exp_A0 = neighbors_front_mmd_exp_A0 + neighbors_back_mmd_exp_A0
  row_neighbors_mmd_sq_A0 = neighbors_front_mmd_sq_A0 + neighbors_back_mmd_sq_A0

  ws2_estimates_A0[i] = est_row_ws2_A0
  mmd_estimates_exp_A0[i] = est_row_mmd_exp_A0
  mmd_estimates_sq_A0[i] = est_row_mmd_sq_A0

  nn_ws2_A0[i] = row_nn_ws2_A0
  nn_mmd_exp_A0[i] = row_nn_mmd_exp_A0
  nn_mmd_sq_A0[i] = row_nn_mmd_sq_A0

  neighbors_ws2_A0[i] = row_neighbors_ws2_A0
  neighbors_mmd_exp_A0[i] = row_neighbors_mmd_exp_A0
  neighbors_mmd_sq_A0[i] = row_neighbors_mmd_sq_A0


User 0
100%|██████████| 100/100 [00:43<00:00,  2.28trial/s, best loss: 30.271724136401925]
Estimating unit 0
User 1
100%|██████████| 100/100 [00:27<00:00,  3.69trial/s, best loss: 30.68707890063335]
Estimating unit 1
User 2
100%|██████████| 100/100 [00:28<00:00,  3.51trial/s, best loss: 28.67193395319203]
Estimating unit 2
User 3
100%|██████████| 100/100 [00:23<00:00,  4.34trial/s, best loss: 59.19771436475567]
Estimating unit 3
User 4
100%|██████████| 100/100 [00:45<00:00,  2.20trial/s, best loss: 18.42746225203476]
Estimating unit 4
User 5
100%|██████████| 100/100 [00:32<00:00,  3.04trial/s, best loss: inf]
Estimating unit 5
User 6
100%|██████████| 100/100 [00:36<00:00,  2.78trial/s, best loss: 28.013085129244796]
Estimating unit 6
User 7
100%|██████████| 100/100 [00:40<00:00,  2.46trial/s, best loss: 37.779821545777224]
Estimating unit 7
User 8
100%|██████████| 100/100 [00:38<00:00,  2.62trial/s, best loss: 34.550445262126004]
Estimating unit 8
User 9
100%|██████████| 100/100 [00:17

c:\Users\Caleb\HeartstepsDNN\Heartsteps-DNN\src\dnn.py:379: RuntimeWarning: Mean of empty slice
  est = np.nanmean(Z_cp[i, :], axis = 0).flatten()



100%|██████████| 100/100 [02:40<00:00,  1.60s/trial, best loss: inf]
Estimating unit 28
User 29
100%|██████████| 100/100 [00:40<00:00,  2.46trial/s, best loss: 133.87322377925636]
Estimating unit 29
User 30
100%|██████████| 100/100 [00:15<00:00,  6.62trial/s, best loss: 50.39980474844358]
Estimating unit 30
User 31
100%|██████████| 100/100 [00:33<00:00,  3.03trial/s, best loss: 13.918335771462925]
Estimating unit 31
User 32
100%|██████████| 100/100 [00:18<00:00,  5.41trial/s, best loss: 24.8462940740715]
Estimating unit 32
User 33
100%|██████████| 100/100 [00:38<00:00,  2.62trial/s, best loss: 10.738263480525879]
Estimating unit 33
User 34
100%|██████████| 100/100 [00:26<00:00,  3.82trial/s, best loss: 21.511809304234824]
Estimating unit 34
User 35
100%|██████████| 100/100 [00:22<00:00,  4.39trial/s, best loss: 55.65856514516679]
Estimating unit 35
User 36
100%|██████████| 100/100 [00:43<00:00,  2.29trial/s, best loss: 21.690028072099146]
Estimating unit 36


In [28]:
# save full estimates, neighbors, nn count
np.save('ws2_fullestimatesA1_v9_log.npy', ws2_estimates_A1) 
np.save('mmd_exp_fullestimatesA1_v9_log.npy', np.array(mmd_estimates_exp_A1,dtype=object), allow_pickle=True) 
np.save('mmd_sq_fullestimatesA1_v9_log.npy', np.array(mmd_estimates_sq_A1,dtype=object), allow_pickle=True) 

np.save('ws2_fullneighborsA1_v9_log.npy', np.array(neighbors_ws2_A1, dtype = object), allow_pickle=True)
np.save('mmd_exp_fullneighborsA1_v9_log.npy', np.array(neighbors_mmd_exp_A1, dtype = object), allow_pickle=True)
np.save('mmd_sq_fullneighborsA1_v9_log.npy', np.array(neighbors_mmd_sq_A1, dtype = object), allow_pickle=True)

np.save('ws2_fullnncountA1_v9_log.npy', nn_ws2_A1) 
np.save('mmd_exp_fullnncountA1_v9_log.npy', nn_mmd_exp_A1)
np.save('mmd_sq_fullnncountA1_v9_log.npy', nn_mmd_sq_A1)


np.save('eta_stars_A1_v9_log.npy', eta_stars_A1)

# A0 Business ------------------------------

np.save('ws2_fullestimatesA0_v9_log.npy', ws2_estimates_A0) 
np.save('mmd_exp_fullestimatesA0_v9_log.npy', np.array(mmd_estimates_exp_A0,dtype=object), allow_pickle=True) 
np.save('mmd_sq_fullestimatesA0_v9_log.npy', np.array(mmd_estimates_sq_A0,dtype=object), allow_pickle=True) 

np.save('ws2_fullneighborsA0_v9_log.npy', np.array(neighbors_ws2_A0, dtype = object), allow_pickle=True)
np.save('mmd_exp_fullneighborsA0_v9_log.npy', np.array(neighbors_mmd_exp_A0, dtype = object), allow_pickle=True)
np.save('mmd_sq_fullneighborsA0_v9_log.npy', np.array(neighbors_mmd_sq_A0, dtype = object), allow_pickle=True)


np.save('ws2_fullnncountA0_v9_log.npy', nn_ws2_A0) 
np.save('mmd_exp_fullnncountA0_v9_log.npy', nn_mmd_exp_A0)
np.save('mmd_sq_fullnncountA0_v9_log.npy', nn_mmd_sq_A0)

np.save('eta_stars_A0_v9_log.npy', eta_stars_A0)

In [36]:
# KTE is -> A1 estimates (mmd) A0 observed and A0 estimates (mmd) A1 observed
# OR KTE is A1 estimates (mmd) A0 estimates

KTE_obvs_exp = np.full([N, T], np.nan)
KTE_ests_exp = np.full([N, T], np.nan)

KTE_obvs_sq = np.full([N, T], np.nan)
KTE_ests_sq = np.full([N, T], np.nan)

KTE_obvs_ks_sq = np.full([N, T], np.nan)
KTE_ests_ks_sq = np.full([N, T], np.nan)

KTE_obvs_ks_exp = np.full([N, T], np.nan)
KTE_ests_ks_exp = np.full([N, T], np.nan)

for i in range(N):
  for j in range(T):
    if A_hs[i, j] != 2:
    # if not mmd_estimates_exp_A0 is None and not mmd_estimates_exp_A1 is None:
      KTE_ests_exp[i, j] = dnn.sqmmd_est2(mmd_estimates_exp_A0[i][j], mmd_estimates_exp_A1[i][j], 
                                            kernel = "exponential")
      KTE_ests_ks_exp[i, j] = kstest(mmd_estimates_exp_A0[i][j], mmd_estimates_exp_A1[i][j]).statistic

      # if not mmd_estimates_sq_A0 is None and not mmd_estimates_sq_A1 is None:  
      KTE_ests_sq[i, j] = dnn.sqmmd_est2(mmd_estimates_sq_A0[i][j], mmd_estimates_sq_A1[i][j],
                                          kernel = "square")
      KTE_ests_ks_sq[i, j] = kstest(mmd_estimates_sq_A0[i][j], mmd_estimates_sq_A1[i][j]).statistic


      if A_hs[i, j] == 0:
        KTE_obvs_exp[i, j] = dnn.sqmmd_est2(mmd_estimates_exp_A1[i][j], M_hs[i, j],
                                            kernel = "exponential")
        KTE_obvs_sq[i, j] = dnn.sqmmd_est2(mmd_estimates_sq_A1[i][j], M_hs[i, j],
                                          kernel = "square")

        KTE_obvs_ks_exp[i, j] = kstest(mmd_estimates_exp_A1[i][j], M_hs[i, j]).statistic
        KTE_obvs_ks_sq[i, j] = kstest(mmd_estimates_sq_A1[i][j], M_hs[i, j]).statistic
      else:
        KTE_obvs_exp[i, j] = dnn.sqmmd_est2(M_hs[i, j], mmd_estimates_exp_A0[i][j],
                                            kernel = "exponential")
        KTE_obvs_sq[i, j] = dnn.sqmmd_est2(M_hs[i, j], mmd_estimates_sq_A0[i][j],
                                          kernel = "square")

        KTE_obvs_ks_exp[i, j] = kstest(M_hs[i, j], mmd_estimates_exp_A0[i][j]).statistic
        KTE_obvs_ks_sq[i, j] = kstest(M_hs[i, j], mmd_estimates_sq_A0[i][j]).statistic


  
  

C:\Users\Caleb\AppData\Local\Temp\ipykernel_31552\3704875382.py:22: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  KTE_ests_ks_exp[i, j] = kstest(mmd_estimates_exp_A0[i][j], mmd_estimates_exp_A1[i][j]).statistic
C:\Users\Caleb\AppData\Local\Temp\ipykernel_31552\3704875382.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  KTE_ests_ks_sq[i, j] = kstest(mmd_estimates_sq_A0[i][j], mmd_estimates_sq_A1[i][j]).statistic
C:\Users\Caleb\AppData\Local\Temp\ipykernel_31552\3704875382.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your arr

In [38]:
np.save("kte_obvs_sq.npy", KTE_obvs_sq)
np.save("kte_obvs_exp.npy", KTE_obvs_exp)
np.save("kte_obvs_ks_sq.npy", KTE_obvs_ks_sq)
np.save("kte_obvs_ks_exp.npy", KTE_obvs_ks_exp)

np.save("kte_ests_exp.npy", KTE_ests_exp)
np.save("kte_ests_sq.npy", KTE_ests_sq)
np.save("kte_ests_ks_exp.npy", KTE_ests_ks_exp)
np.save("kte_ests_ks_sq.npy", KTE_ests_ks_sq)



In [85]:
fig, (ax1, ax2) = plt.subplots(figsize=(16, 8), nrows = 2)


pos1 = ax1.imshow(KTE_obvs_exp, aspect = 'auto')
pos2 = ax2.imshow(KTE_obvs_ks_exp, aspect = 'auto')

fig.colorbar(pos1, ax = ax1, label = "KTE$_{gaussian}$")
fig.colorbar(pos2, ax = ax2, label = "KS-test treatment effect")

ax2.set_xlabel("Decision times")
ax2.set_ylabel("Units")

ax1.set_ylabel("Units")

ax1.get_xaxis().set_ticklabels([])
ax2.get_xaxis().set_ticklabels([])

ax1.set_title("Estimated Treatment Effect - Kernel-NN$_{gaussian}$")
#ax2.set_title("Kolmogorov-Smirnov test treatment effect")
#ax3.set_title("Kernel-NN with square kernel")

#fig.suptitle("2-sample Kolmogorov-Smirnov test statistic by DNN method")
plt.savefig("kte_ks_exp.pdf", bbox_inches = "tight")

In [88]:
# unit 0



fig, ax = plt.subplots(figsize=(8,8))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.4)
ax.scatter(x = np.arange(len(KTE_obvs_exp[0])), y = KTE_obvs_exp[0], alpha= 0.8, color = "darkgreen")
ax.set_xlabel("Day in trial")
ax.set_ylabel("Estimated KTE$_{Gaussian}$")
ax.set_title("Estimated Kernel Treatment Effect - Unit 0")

plt.savefig("kte_unit0_exp.pdf", bbox_inches = "tight")


In [89]:
fig, ax = plt.subplots(figsize=(8,8))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.4)
ax.scatter(x = np.arange(len(KTE_obvs_exp[0])), y = KTE_obvs_ks_exp[0], alpha= 0.8, color = "goldenrod")
ax.set_xlabel("Day in trial")
ax.set_ylabel("Kolmogorov-Smirnov test statistic")
ax.set_title("KS-test Treatment Effect - Unit 0")

plt.savefig("kte_unit0_ks.pdf", bbox_inches = "tight")